In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!ls /kaggle/input


In [ ]:
import numpy as np
import random
import os
import zipfile
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
import os

print(os.listdir("/kaggle/input"))


In [ ]:
import os

dataset_path = "/kaggle/input/optimization-route-planning-optimal"
print(os.listdir(dataset_path))


In [ ]:
extract_path = "/kaggle/input/optimization-route-planning-optimal"
files = sorted(os.listdir(extract_path))

print("Total datasets:", len(files))
files[:5]


In [ ]:
selected_files = files[:10]
selected_files


In [ ]:
def read_dataset(path):
    customers = []
    with open(path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 3 and parts[0].isdigit():
                customers.append((
                    int(parts[0]),
                    float(parts[1]),
                    float(parts[2])
                ))
    return customers


In [ ]:
def euclidean(a, b):
    return np.sqrt((a[1]-b[1])**2 + (a[2]-b[2])**2)

def traffic_factor():
    return random.choice([0.8, 1.0, 1.5])  # low, normal, peak


In [ ]:
def ga_fitness_normal(route, customers):
    return sum(
        euclidean(customers[route[i]], customers[route[i+1]])
        for i in range(len(route)-1)
    )

def ga_fitness_traffic(route, customers):
    return sum(
        euclidean(customers[route[i]], customers[route[i+1]]) * traffic_factor()
        for i in range(len(route)-1)
    )


In [ ]:
def ga_population(size, n):
    pop = []
    for _ in range(size):
        route = list(range(1, n))
        random.shuffle(route)
        pop.append([0] + route + [0])
    return pop


In [ ]:
def ga_selection(pop, fitness, customers):
    selected = []
    for _ in range(len(pop)):
        a, b = random.sample(pop, 2)
        selected.append(a if fitness(a, customers) < fitness(b, customers) else b)
    return selected


In [ ]:
def ga_selection(pop, fitness, customers):
    selected = []
    for _ in range(len(pop)):
        a, b = random.sample(pop, 2)
        selected.append(a if fitness(a, customers) < fitness(b, customers) else b)
    return selected


In [ ]:
def ga_crossover(p1, p2):
    start, end = sorted(random.sample(range(1, len(p1)-1), 2))
    child = [-1]*len(p1)
    child[start:end] = p1[start:end]

    ptr = 1
    for g in p2:
        if g not in child:
            while child[ptr] != -1:
                ptr += 1
            child[ptr] = g

    child[0] = child[-1] = 0
    return child


In [ ]:
def ga_mutation(route, rate=0.1):
    if random.random() < rate:
        i, j = random.sample(range(1, len(route)-1), 2)
        route[i], route[j] = route[j], route[i]
    return route


In [ ]:
def run_ga(customers, fitness_fn, gens=120, pop_size=40):
    pop = ga_population(pop_size, len(customers))
    history = []

    for _ in range(gens):
        pop = ga_selection(pop, fitness_fn, customers)
        new_pop = []
        for i in range(0, pop_size, 2):
            c1 = ga_crossover(pop[i], pop[i+1])
            c2 = ga_crossover(pop[i+1], pop[i])
            new_pop.append(ga_mutation(c1))
            new_pop.append(ga_mutation(c2))
        pop = new_pop
        history.append(min(fitness_fn(r, customers) for r in pop))

    return history[-1], history


In [ ]:
def run_aco(customers, ants=20, iters=100, alpha=1, beta=2, rho=0.5):
    n = len(customers)
    dist = np.zeros((n,n))
    for i in range(n):
        for j in range(n):
            dist[i][j] = euclidean(customers[i], customers[j])

    pheromone = np.ones((n,n))
    best_cost = float("inf")
    history = []

    for _ in range(iters):
        for _ in range(ants):
            route = [0]
            visited = set(route)

            while len(route) < n:
                i = route[-1]
                probs = []
                for j in range(n):
                    if j not in visited:
                        probs.append((pheromone[i][j]**alpha) * ((1/dist[i][j])**beta))
                    else:
                        probs.append(0)
                probs = np.array(probs) / sum(probs)
                j = np.random.choice(range(n), p=probs)
                route.append(j)
                visited.add(j)

            route.append(0)
            cost = sum(dist[route[i]][route[i+1]] * traffic_factor() for i in range(len(route)-1))
            best_cost = min(best_cost, cost)

        pheromone *= (1-rho)
        history.append(best_cost)

    return best_cost, history


In [ ]:
import time

results = []
start_all = time.time()

for idx, f in enumerate(selected_files, 1):
    start = time.time()
    
    customers = read_dataset(f"{extract_path}/{f}")
    if len(customers) < 5:
        continue

    ga_val, ga_hist = run_ga(customers, ga_fitness_traffic)
    aco_val, aco_hist = run_aco(customers)

    results.append({
        "Dataset": f,
        "GA_Traffic": ga_val,
        "ACO_Traffic": aco_val
    })

    print(f"[{idx}/{len(selected_files)}] {f} completed in {round(time.time()-start,2)} sec")

print("Total time:", round(time.time() - start_all, 2), "seconds")


In [ ]:
df = pd.DataFrame(results)
df


In [ ]:
df.mean(numeric_only=True)


In [ ]:
df.plot(x="Dataset", y=["GA_Traffic", "ACO_Traffic"], kind="bar", figsize=(12,5))
plt.ylabel("Travel Time")
plt.title("Traffic-Aware GA vs ACO")
plt.show()
